1. The Target Architecture

First, we need to move from a single `.ipynb` file to a structured repository. A standard, modular machine learning project separates configuration, data processing, and model logic:

```text
my_ml_project/
├── data/                  # Ignored by git (raw and processed data)
├── notebooks/             # Exploratory only (e.g., 01_eda.ipynb)
├── src/                   # The actual Python package
│   ├── __init__.py
│   ├── config.py          # Centralized hyperparameters and paths
│   ├── data.py            # Data loading and cleaning classes
│   ├── features.py        # Feature engineering logic
│   ├── model.py           # Model definition and training classes
│   └── evaluate.py        # Metrics and validation
├── tests/                 # Unit tests for your python modules
├── pyproject.toml         # Dependency and project metadata
└── train.py               # The main executable script tying it all together

```

2. Dependency Management with Poetry

Relying on a simple `requirements.txt` and `pip` can lead to "dependency hell" because it doesn't strictly lock sub-dependencies, making builds non-deterministic. Instead, modern Python projects use `pyproject.toml` managed by tools like **Poetry** to lock exact versions of every package.

1. Initialize the project: Run in your project root folder.
Run `poetry init` in your terminal. This interactive prompt will generate a `pyproject.toml` file containing your project metadata.


2. Add production dependencies:
Instead of `pip install`, lock your core ML libraries so they are permanently version-controlled:

```bash
poetry add pandas scikit-learn xgboost

```


3. Add development dependencies: Tools for coding, not production.
Keep testing and formatting tools completely separate from your production build to keep your eventual Docker image lightweight:

```bash
poetry add --group dev pytest ruff jupyter

```


4. Execute in the isolated environment:
Run your code within the guaranteed environment using `poetry run python train.py`, or activate the virtual shell directly with `poetry shell`.


3. Refactoring: From Cells to Classes

The goal of refactoring is to make your code 'modular' (each piece does one specific thing) and **object-oriented** (using classes to manage state).

Extract Configuration

Hardcoded paths and hyperparameters in notebooks cause silent failures in production. Move them to a dedicated `config.py` using Python's `dataclasses`.

In [ ]:
# src/config.py
from dataclasses import dataclass

@dataclass
class ModelConfig:
    data_path: str = "data/raw/dataset.csv"
    test_size: float = 0.2
    random_state: int = 42
    n_estimators: int = 100

Build Object-Oriented Data Pipelines


In [ ]:
# src/data.py
import pandas as pd
from sklearn.model_selection import train_test_split

class DataLoader:
    def __init__(self, config):
        self.config = config

    def load_and_split(self):
        df = pd.read_csv(self.config.data_path)

        # Apply cleaning and feature engineering...
        X = df.drop("target", axis=1)
        y = df["target"]

        return train_test_split(
            X, y,
            test_size=self.config.test_size,
            random_state=self.config.random_state
        )

### The Model Wrapper



In [ ]:
# src/model.py
from sklearn.ensemble import RandomForestClassifier

class ModelTrainer:
    def __init__(self, config):
        self.config = config
        self.model = RandomForestClassifier(
            n_estimators=self.config.n_estimators,
            random_state=self.config.random_state
        )

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        return self.model

## 4. The Final Pipeline



In [ ]:
# train.py
from src.config import ModelConfig
from src.data import DataLoader
from src.model import ModelTrainer

def main():
    # 1. Load configuration
    config = ModelConfig()

    # 2. Process data
    loader = DataLoader(config)
    X_train, X_test, y_train, y_test = loader.load_and_split()

    # 3. Train model
    trainer = ModelTrainer(config)
    trained_model = trainer.train(X_train, y_train)

    print("Training pipeline completed successfully.")

if __name__ == "__main__":
    main()